# Task 2: Holistic gRNA Design for CRISPRa

**Goal**: Systematically design a multiplexed pool of Guide RNAs for the CRISPRa activation of our primary target gene.

### Changelogs:
1. **Biological Padding:** Target the first exon, but include a ±25bp biological padding to allow for proper protospacer overlap at the exon edges.
2. **Multiplexing:** Select a pool of the top n guides rather than a single guide, as CRISPRa is highly synergistic when multiple activators bind.
3. **Resilient Filtering:** Use a weighted composite score (efficiency + specificity) to rank guides reasonably.


In [1]:
import gzip
import os

import matplotlib.pyplot as plt
import pandas as pd
import requests
from IPython.display import display

# Ensure output directories exist
os.makedirs('../data', exist_ok=True)
os.makedirs('../results/tables', exist_ok=True)

## 1. Target Definition (Task 1 Consensus)

Based on the `consensus_de_genes_strict.csv` output from Task 1, there were exactly **8 genes** that passed the strict statistical thresholds to be classified as "Consensus DE". 

Task 2 requires designing a guide RNA for **CRISPRa (Activation)**, our target gene *must* be **downregulated** (possessing a negative Log2 Fold Change) so that the CRISPRa system can restore its expression. 

If we look at the 8 consensus genes and their average Log2 Fold Changes (LFC):
1. `OAS2` (LFC: +9.25)
2. `GPRASP1` (LFC: +9.01)
3. `AC026801.2` (LFC: +8.49)
4. `LPL` (LFC: +8.39)
5. `XAF1` (LFC: +6.44)
6. `CXCL11` (LFC: +5.79)
7. `IFI27` (LFC: +3.75)
8. **`SERPINB2` (LFC: -1.43)**

**`SERPINB2` (Chromosome 18)** is the *only* gene on the consensus list with a negative LFC. Every other significant gene is heavily upregulated. Therefore, `SERPINB2` is the only mathematically and biologically viable target for this CRISPRa pipeline.

In [2]:
target_gene = 'SERPINB2'
target_chrom = '18'
chrom_accession = 'NC_000018.10'

print(f"Targeting: {target_gene} (Chromosome {target_chrom})")

Targeting: SERPINB2 (Chromosome 18)


## 2. Sequence Extraction (First Exon + Padding)

To ensure our 20nt guide RNAs can properly overlap the edges of the first exon without falling outside the search space, we include a ±25bp flanking padding during sequence extraction.

In [3]:
gff_file = '../GCF_000001405.26_GRCh38_genomic.gff.gz'
target_exons = []

# Parse the GFF file to locate the exact coordinates of the target gene's exons
with gzip.open(gff_file, 'rt') as file:
    for line in file:
        if line.startswith('#'):
            continue
            
        # FIX 1: Strict attribute matching (gene=SERPINB2;) prevents silently grabbing unrelated features or isoforms
        # We also specifically filter for transcript variant 1 (NM_001143818.1) as the canonical representative
        if '\texon\t' in line and f'gene={target_gene};' in line and 'NM_001143818.1' in line:
            parts = line.strip().split('\t')
            if parts[0] == chrom_accession:
                target_exons.append({
                    'start': int(parts[3]), 
                    'end': int(parts[4]), 
                    'strand': parts[6]
                })

exons_df = pd.DataFrame(target_exons).drop_duplicates().reset_index(drop=True)

if not exons_df.empty:
    strand = exons_df['strand'].iloc[0]
    
    # Determine the first exon based on the coding strand
    if strand == '-':
        first_exon = exons_df.sort_values(by='end', ascending=False).iloc[0]
    else:
        first_exon = exons_df.sort_values(by='start', ascending=True).iloc[0]
        
    # We save the unpadded boundaries globally to verify our guides later!
    unpadded_start = first_exon['start']
    unpadded_end = first_exon['end']
        
    # Apply ±25bp flanking biological padding for the search space
    start_pos = unpadded_start - 25
    end_pos = unpadded_end + 25

    print(f"True First Exon located at: {unpadded_start} - {unpadded_end} (Strand {strand})")
    print(f"Extracting Search Space (with ±25bp padding): {start_pos} - {end_pos}")

    # Fetch the padded sequence from the Ensembl REST API
    server = "https://rest.ensembl.org"
    endpoint = f"/sequence/region/human/{target_chrom}:{start_pos}..{end_pos}:{1 if strand == '+' else -1}"
    
    response = requests.get(server + endpoint, headers={"Content-Type": "text/plain"})
    
    if response.ok:
        fasta_path = f"../data/{target_gene.lower()}_first_exon_padded.fasta"
        with open(fasta_path, 'w') as file:
            file.write(f">{target_gene}_first_exon_chr{target_chrom}_{start_pos}_{end_pos}\n")
            file.write(response.text)
            
        # Also fetch the STRICT unpadded sequence for verification later
        unpad_resp = requests.get(server + f"/sequence/region/human/{target_chrom}:{unpadded_start}..{unpadded_end}:{1 if strand == '+' else -1}", headers={"Content-Type": "text/plain"})
        unpadded_seq = unpad_resp.text        
        print(f"\n--- Unpadded First Exon Sequence ({unpadded_end - unpadded_start + 1} bp) ---")
        print(unpadded_seq)
        print("-" * 50)
            
        print(f"Success! {len(response.text)} bp padded sequence saved to {fasta_path}")
    else:
        print(f"API Fetch Failed. HTTP {response.status_code}: {response.text}")
else:
    print(f"Error: No exons found for {target_gene}.")


True First Exon located at: 63887705 - 63887770 (Strand +)
Extracting Search Space (with ±25bp padding): 63887680 - 63887795



--- Unpadded First Exon Sequence (66 bp) ---
GTAACAACTCTCAGAGGAGCATTGCCCGTCAGACAGCAACTCAGAGAATAACCAGAGAACAACCAG
--------------------------------------------------
Success! 116 bp padded sequence saved to ../data/serpinb2_first_exon_padded.fasta


## 3. Guide Evaluation & Multiplex Pool Selection

### Tool Choice Justification (Why CRISPOR?)
While tools like CHOPCHOP are popular, we explicitly chose **CRISPOR** for this pipeline because it calculates the advanced **CFD (Cutting Frequency Determination) Specificity Score**. Basic tools just count the raw number of mismatches (e.g., "0 off-targets with 1 mismatch"). However, the CFD algorithm biologically understands that a mismatch near the PAM site destroys binding entirely, whereas a mismatch 18bp away is easily tolerated. By using CRISPOR, our algorithm can mathematically penalize off-targets based on their *position* and *type*, yielding a vastly superior safety metric.

### Safety Filtering Explanation
CRISPRa demands extremely high specificity. Unintended off-target binding could accidentally activate oncogenes or radically alter cell states. 
To prevent this, we apply a **strict base safety filter (CFD > 50 and MIT > 50)** to immediately discard dangerous guides (a CFD of 20-30 is far too permissive for a safe CRISPR experiment). We then rank the surviving guides using a **Composite Score** weighted 60% towards Safety (CFD Specificity) and 40% towards Efficiency (Doench '16). This ensures our multiplex pool heavily prioritizes biological safety while maintaining robust activation.


In [4]:
# Load the external CRISPOR genome-wide evaluation results
results_file = f'../data/crispor_{target_gene.lower()}_results.xls'
guides_df = pd.read_excel(results_file, header=8) 

col_map = {
    '#guideId': '#guideId', 
    'mitSpecScore': 'mitSpecScore', 
    'cfdSpecScore': 'cfdSpecScore', 
    "Doench '16-Score": 'Efficiency'
}
guides_df = guides_df.rename(columns=col_map)

# Extract strand information from the guide ID
guides_df['strand'] = guides_df['#guideId'].apply(lambda x: '+' if 'forw' in x else '-')

# 1. Base Safety Filter (Industry Standard CFD > 50)
mit_mask = guides_df['mitSpecScore'] > 50
cfd_mask = guides_df['cfdSpecScore'] > 50
safe_guides = guides_df[mit_mask & cfd_mask].copy()

# 2. Strict Cut-Site Boundary Filter (Tutor's Clarification)
# The tutor explicitly clarified: "cut being in the first exon but the entire sequence doesn't have to be".
# We will calculate the exact genomic cut site of the guide and ensure it falls within the unpadded 66bp exon.
def rev_comp(seq):
    comp = {'A':'T', 'T':'A', 'C':'G', 'G':'C', 'N':'N'}
    return "".join(comp.get(b, 'N') for b in reversed(seq))

with open(f"../data/{target_gene.lower()}_first_exon_padded.fasta", 'r') as f:
    padded_seq = "".join(l.strip() for l in f.readlines()[1:])
    
padding_length = 25
exon_length = len(padded_seq) - (2 * padding_length)

def is_cut_inside_exon(row):
    seq = row['targetSeq']
    strand = row['strand']
    
    if strand == '+':
        idx = padded_seq.find(seq)
        if idx == -1: return False
        cut_site = idx + 17  # Cut site is 3bp upstream of NGG
    else:
        rev = rev_comp(seq)
        idx = padded_seq.find(rev)
        if idx == -1: return False
        cut_site = idx + 6   # Cut site is 3bp downstream of CCN on the opposing strand
        
    # Check if the cut site falls inside the strict unpadded exon coordinates
    return padding_length <= cut_site < (padding_length + exon_length)

safe_guides['Valid_Cut_Site'] = safe_guides.apply(is_cut_inside_exon, axis=1)
valid_guides = safe_guides[safe_guides['Valid_Cut_Site'] == True].copy()

print(f"Of the {len(safe_guides)} safe guides, {len(valid_guides)} have their actual cut site inside the first exon.")

# 3. Composite Weighted Score (40% Efficiency, 60% Safety)
valid_guides['Composite_Score'] = (valid_guides['Efficiency'] * 0.4) + (valid_guides['cfdSpecScore'] * 0.6)

# 4. Final Selection
top_pool = valid_guides.sort_values(by='Composite_Score', ascending=False).head(4)

print(f"\nSelected optimal guides for CRISPRa Multiplex Pool:")
display(top_pool[['#guideId', 'targetSeq', 'Efficiency', 'cfdSpecScore', 'Composite_Score']])

out_csv = f'../results/tables/task2_multiplex_pool_{target_gene.lower()}.csv'
top_pool.to_csv(out_csv, index=False)


Of the 7 safe guides, 5 have their actual cut site inside the first exon.

Selected optimal guides for CRISPRa Multiplex Pool:


,#guideId,targetSeq,Efficiency,cfdSpecScore,Composite_Score
4,40forw,TGAACTGTAACAACTCTCAGAGG,68,83,77.0
6,90forw,AGAATAACCAGAGAACAACCAGG,71,80,76.4
3,50rev,TCTCTGAGTTGCTGTCTGACGGG,51,83,70.2
2,77rev,TGAAATACCTGGTTGTTCTCTGG,37,85,65.8


## 4. Final Selection & Validation Checks

**Final Selection:**
The pipeline successfully identified an optimal multiplex pool of 4 guides, topped by **`AGAATAACCAGAGAACAACC AGG`** (Composite Score: **76.4**) and **`TGAACTGTAACAACTCTCAG AGG`** (Composite Score: **77.0**). These mathematically outperform manual single-guide selections by maximizing Doench '16 Efficiency while maintaining rigorous CFD Specificity safety constraints.